# 3 . EMA-Cross Variant Study

Systematically answers the exploration questions for the price-vs-EMA strategy by **sweeping**
each variant with the fast engine. Base idea: long above the Nth EMA, short below - with every
variant a parameter of `EmaCross`.

**Method:** to compare *entries* fairly (Q1-Q6) we hold the EXIT fixed (SL/TP from the config) so
only the entry changes. Then we study *exits* (Q7-Q9) on a fixed good entry. Each cell prints the
top settings by **Sharpe**; see the Metrics section at the bottom for how to judge them.


## Setup & data
Intraday short-term -> default **5-minute** gold. Use a modest window while exploring; widen for
the final confirmation. Costs are ON (gold spread) - intraday edges live or die on costs.


In [2]:
import sys, os, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import dataclasses, pandas as pd
from quant.data import get_ohlcv
from quant.engine import BacktestConfig
from quant.strategies import EmaCross
from quant import analytics as A
from quant.viz import sweep_heatmap
from experiments.base import Experiment

SYMBOL, SOURCE, MARKET = 'PAXGUSDT', 'binance', 'spot'   # or 'XAUUSD' + SOURCE='dukascopy'
TF          = '5m'
START, END  = '2026-01-01', '2026-07-31'
df = get_ohlcv(SYMBOL, TF, start=START, end=END, source=SOURCE, market=MARKET, tz='UTC')
print(f'{len(df):,} {TF} bars  {df.open_time.min()} -> {df.open_time.max()}')

[05:42:22] INFO | quant.data | cache EXTEND binance PAXGUSDT 5m -> incremental fetch
[05:42:22] INFO | 01. Parameters: market=spot, symbol=PAXGUSDT, interval=5m, start=2026-01-01 00:00:00+00:00, end=2026-07-31 00:00:00+00:00
[05:42:22] INFO | 02. Cache directory: C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\data (cwd=C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks)
[05:42:22] INFO | 03. Main cache file: C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\data\binance_spot_PAXGUSDT_5m.parquet
[05:42:22] INFO | 04. Partial checkpoint dir: C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\data\.partials\binance_spot_PAXGUSDT_5m
[05:42:22] INFO | 05. Cache: HIT | rows=163,302 (main_rows=163,302, partial_files=0) | range=2025-01-01 00:00:00+00:00 -> 2026-07-22 00:25:00+00:00
[05:42:22] INFO | 06. Fetch plan 1/1: 2026-07-22 00:30:00+00:00 -> 2026-07-31 00:00:00+00:00
[05:42:22] INFO | 07. Starting fetch range 1/1.


PAXGUSDT 5m:   0%|          | 0/3 [00:00<?, ?page/s]

[05:42:24] INFO | 08. Completed fetch range 1/1 | rows=3 | total_rows_fetched=3 | requests=1 | retries=0 | elapsed=1.5s
[05:42:24] INFO | 09. Cache consolidated and saved (parquet) | rows=163,305 | range=2025-01-01 00:00:00+00:00 -> 2026-07-22 00:40:00+00:00 | cleared_partial_files=1
[05:42:24] INFO | 10. Fetch summary | new_rows=3 | successful_pages=1 | requests=1 | retries=0 | partial_saves=1 | elapsed=1.8s
[05:42:24] INFO | 11. Returning slice | rows=58,185 | range=2026-01-01 00:00:00+00:00 -> 2026-07-22 00:40:00+00:00


58,185 5m bars  2026-01-01 00:00:00+00:00 -> 2026-07-22 00:40:00+00:00


In [3]:
# baseline execution model: risk 1% per trade, SL 0.6%, TP 2R, realistic gold spread
base_cfg = BacktestConfig(initial_cash=10_000, fee_bps=0, slippage_bps=1.0, spread=0.24,
                          exit_enabled=True, sl_mode='entry_pct', sl_value=0.6,
                          tp_mode='rr', tp_value=2.0, sizing_mode='risk_pct_equity',
                          sizing_value=1.0, allow_short=True)
SHOW = ['num_trades','total_return_pct','win_rate_pct','profit_factor','sharpe','max_drawdown_pct']

def study(name, desc, strat_space=None, cfg_space=None, metric='sharpe', min_trades=30, valid_fn=None, cfg=None):
    # run one experiment on the loaded df; returns the ranked DataFrame
    exp = Experiment(name=name, description=desc, strategy_cls=EmaCross, base_cfg=cfg or base_cfg,
                     symbol=SYMBOL, tf=TF, start=START, end=END,
                     strategy_space=strat_space or {}, cfg_space=cfg_space or {},
                     metric=metric, direction='max', min_trades=min_trades, valid_fn=valid_fn)
    return exp.run(df=df)

## Q1 . Entry trigger: cross vs close-above vs full-candle-above
Does requiring a *close* above the EMA, or a *whole candle* above it, beat a bare *cross*?


In [4]:
r = study('q1_entry_mode', 'cross vs close vs full_candle entry',
          strat_space={'entry_mode':['cross','close','full_candle'], 'confirm_n':[1], 'exit_mode':['none']})
r[['entry_mode']+SHOW]

[05:42:47] INFO | experiments | experiment 'q1_entry_mode': 3 strategy x 1 cfg = 3 trials | bars=58185
[05:42:47] INFO | experiments | experiment 'q1_entry_mode' -> experiments\results\q1_entry_mode (best sharpe=0.4955)


,entry_mode,num_trades,total_return_pct,win_rate_pct,profit_factor,sharpe,max_drawdown_pct
0,close,694.0,6.148563,34.726225,1.019907,0.495452,16.558099
1,full_candle,689.0,-5.179534,33.817126,0.982645,-0.139254,22.883435
2,cross,365.0,-11.584149,32.328767,0.917422,-0.867495,17.398570


## Q2 . Confirmation candles + colour
How many candles must hold the condition, and must they be the right colour (green for long)?


In [5]:
r = study('q2_confirm', 'confirmation candle count + colour',
          strat_space={'entry_mode':['close','full_candle'], 'confirm_n':[1,2,3,5],
                       'confirm_color':[False,True], 'exit_mode':['none']})
r[['entry_mode','confirm_n','confirm_color']+SHOW].head(12)

[05:43:28] INFO | experiments | experiment 'q2_confirm': 16 strategy x 1 cfg = 16 trials | bars=58185
[05:43:29] INFO | experiments | experiment 'q2_confirm' -> experiments\results\q2_confirm (best sharpe=1.7142)


,entry_mode,confirm_n,confirm_color,num_trades,total_return_pct,win_rate_pct,profit_factor,sharpe,max_drawdown_pct
0,close,3,True,546.0,28.054610,36.813187,1.105029,1.714159,12.264095
1,full_candle,2,True,594.0,18.479955,35.858586,1.064844,1.178845,13.986525
2,close,2,True,615.0,15.989762,35.609756,1.055568,1.029834,15.026994
3,full_candle,2,False,665.0,7.488011,34.887218,1.025265,0.569701,15.247606
4,full_candle,1,True,637.0,7.335260,34.850863,1.024823,0.566280,21.666336
5,close,1,False,694.0,6.148563,34.726225,1.019907,0.495452,16.558099
6,full_candle,3,False,639.0,5.359731,34.741784,1.019220,0.458535,19.497558
7,close,3,False,705.0,2.434504,34.468085,1.008139,0.296606,14.120795
8,full_candle,3,True,533.0,2.226913,34.521576,1.009234,0.283190,19.982711
9,close,2,False,703.0,-1.342357,34.139403,0.995468,0.086683,17.111697


## Q3 . Heikin-Ashi vs regular candles


In [6]:
r = study('q3_heikin', 'regular vs Heikin-Ashi',
          strat_space={'use_heikin_ashi':[False,True], 'entry_mode':['close','full_candle'],
                       'confirm_n':[2,3], 'exit_mode':['none']})
r[['use_heikin_ashi','entry_mode','confirm_n']+SHOW].head(10)

[05:43:41] INFO | experiments | experiment 'q3_heikin': 8 strategy x 1 cfg = 8 trials | bars=58185
[05:43:41] INFO | experiments | experiment 'q3_heikin' -> experiments\results\q3_heikin (best sharpe=0.7295)


,use_heikin_ashi,entry_mode,confirm_n,num_trades,total_return_pct,win_rate_pct,profit_factor,sharpe,max_drawdown_pct
0,True,full_candle,3,649.0,10.359964,35.130971,1.035909,0.729486,16.080610
1,False,full_candle,2,665.0,7.488011,34.887218,1.025265,0.569701,15.247606
2,False,full_candle,3,639.0,5.359731,34.741784,1.019220,0.458535,19.497558
3,False,close,3,705.0,2.434504,34.468085,1.008139,0.296606,14.120795
4,False,close,2,703.0,-1.342357,34.139403,0.995468,0.086683,17.111697
5,True,full_candle,2,664.0,-2.881443,34.036145,0.989886,-0.008976,19.153495
6,True,close,2,702.0,-4.255822,33.903134,0.985541,-0.082515,20.076698
7,True,close,3,701.0,-7.414407,33.666191,0.974879,-0.272959,22.227061


## Q4 . Best time of day / weekday (and what to avoid)


In [7]:
base = EmaCross(ema_period=50, entry_mode='full_candle', confirm_n=2, exit_mode='none')
res = base.backtest(df, base_cfg)
print('By session:'); display(A.by_session(res.trades))
bh = A.by_hour(res.trades).sort_values('total_pnl', ascending=False)
print('Best & worst hours:'); display(pd.concat([bh.head(4), bh.tail(4)]))
display(A.by_weekday(res.trades))

[05:43:51] INFO | quant.engine | backtest done | bars=58,185 trades=665 final_cash=10748.80 return=7.49% dd=15.25% | 14.0 ms


By session:


,session,n_trades,win_rate_pct,total_pnl,avg_pnl,profit_factor
0,london,267,32.958801,-600.138928,-2.247711,0.951495
1,newyork,291,31.615120,-1559.140140,-5.357870,0.885342
2,tokyo,242,37.603306,1531.362939,6.327946,1.147846
3,sydney,238,38.655462,1970.830121,8.280799,1.197975


Best & worst hours:


,hour,n_trades,win_rate_pct,total_pnl,avg_pnl,profit_factor
5,5,23,56.521739,1060.185441,46.095019,2.574010
14,14,43,44.186047,914.033975,21.256604,1.553985
11,11,28,46.428571,683.422918,24.407961,1.641655
6,6,33,39.393939,399.506751,12.106265,1.293022
16,16,30,23.333333,-646.458533,-21.548618,0.579750
7,7,17,11.764706,-813.052451,-47.826615,0.232911
18,18,33,21.212121,-925.553983,-28.047090,0.468777
13,13,44,20.454545,-1211.045044,-27.523751,0.504058


,weekday,n_trades,win_rate_pct,total_pnl,avg_pnl,profit_factor,day
0,0,143,28.671329,-1635.795545,-11.439130,0.767644,Mon
1,1,119,41.176471,1617.994003,13.596588,1.337087,Tue
2,2,100,32.000000,-456.091890,-4.560919,0.901551,Wed
3,3,127,35.433071,239.920502,1.889138,1.042782,Thu
4,4,121,35.537190,363.021671,3.000179,1.068840,Fri
5,5,21,52.380952,770.395150,36.685483,2.105240,Sat
6,6,34,32.352941,-150.642826,-4.430671,0.905057,Sun


In [8]:
r = study('q4_session', 'best trading session', metric='total_return_pct',
          strat_space={'entry_mode':['full_candle'], 'confirm_n':[2], 'exit_mode':['none'],
                       'session':[None,'london','newyork','tokyo','sydney']})
r[['session']+SHOW]

[05:43:52] INFO | experiments | experiment 'q4_session': 5 strategy x 1 cfg = 5 trials | bars=58185
[05:43:52] INFO | experiments | experiment 'q4_session' -> experiments\results\q4_session (best total_return_pct=30.0847)


,session,num_trades,total_return_pct,win_rate_pct,profit_factor,sharpe,max_drawdown_pct
0,sydney,355.0,30.084727,38.309859,1.178095,2.199160,8.156408
1,tokyo,379.0,16.924766,36.675462,1.105472,1.350662,14.953898
2,None,665.0,7.488011,34.887218,1.025265,0.569701,15.247606
3,london,402.0,0.011597,34.328358,1.000072,0.123752,20.481521
4,newyork,439.0,-21.768369,31.207289,0.868254,-1.700286,22.627579


## Q5 . Best timeframe (1m / 5m / 15m / 1h)
Higher timeframes usually cut cost-drag and chop; find the sweet spot for short-term trades.


In [9]:
rows=[]
for tf in ['1m','5m','15m','1h']:
    d = get_ohlcv(SYMBOL, tf, start=START, end=END, source=SOURCE, market=MARKET, tz='UTC')
    st = EmaCross(ema_period=50, entry_mode='full_candle', confirm_n=2, exit_mode='none').backtest(d, base_cfg).stats
    rows.append({'tf':tf,'bars':len(d),'trades':st['num_trades'],'return%':round(st['total_return_pct'],1),
                 'sharpe':round(st['sharpe'],2),'maxDD%':round(st['max_drawdown_pct'],1),'PF':round(st['profit_factor'],3)})
pd.DataFrame(rows)

[05:44:23] INFO | quant.data | cache EXTEND binance PAXGUSDT 1m -> incremental fetch
[05:44:23] INFO | 01. Parameters: market=spot, symbol=PAXGUSDT, interval=1m, start=2026-01-01 00:00:00+00:00, end=2026-07-31 00:00:00+00:00
[05:44:23] INFO | 02. Cache directory: C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\data (cwd=C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks)
[05:44:23] INFO | 03. Main cache file: C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\data\binance_spot_PAXGUSDT_1m.parquet
[05:44:23] INFO | 04. Partial checkpoint dir: C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\data\.partials\binance_spot_PAXGUSDT_1m
[05:44:23] INFO | 05. Cache: HIT | rows=816,501 (main_rows=816,501, partial_files=0) | range=2025-01-01 00:00:00+00:00 -> 2026-07-22 00:20:00+00:00
[05:44:23] INFO | 06. Fetch plan 1/1: 2026-07-22 00:21:00+00:00 -> 2026-07-31 00:00:00+00:00
[05:44:23] INFO | 07. Starting fetch range 1/1.


PAXGUSDT 1m:   0%|          | 0/13 [00:00<?, ?page/s]

[05:44:25] INFO | 08. Completed fetch range 1/1 | rows=24 | total_rows_fetched=24 | requests=1 | retries=0 | elapsed=1.7s
[05:44:26] INFO | 09. Cache consolidated and saved (parquet) | rows=816,525 | range=2025-01-01 00:00:00+00:00 -> 2026-07-22 00:44:00+00:00 | cleared_partial_files=1
[05:44:26] INFO | 10. Fetch summary | new_rows=24 | successful_pages=1 | requests=1 | retries=0 | partial_saves=1 | elapsed=2.5s
[05:44:26] INFO | 11. Returning slice | rows=290,925 | range=2026-01-01 00:00:00+00:00 -> 2026-07-22 00:44:00+00:00
[05:44:26] INFO | quant.engine | backtest done | bars=290,925 trades=730 final_cash=12388.36 return=23.88% dd=14.63% | 129.3 ms
[05:44:26] INFO | quant.data | cache EXTEND binance PAXGUSDT 5m -> incremental fetch
[05:44:26] INFO | 01. Parameters: market=spot, symbol=PAXGUSDT, interval=5m, start=2026-01-01 00:00:00+00:00, end=2026-07-31 00:00:00+00:00
[05:44:26] INFO | 02. Cache directory: C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\data (cwd=C:\Users\Tala

PAXGUSDT 5m:   0%|          | 0/3 [00:00<?, ?page/s]

[05:44:28] INFO | 08. Completed fetch range 1/1 | rows=0 | total_rows_fetched=0 | requests=1 | retries=0 | elapsed=1.5s
[05:44:28] INFO | 09. Cache consolidated and saved (parquet) | rows=163,305 | range=2025-01-01 00:00:00+00:00 -> 2026-07-22 00:40:00+00:00 | cleared_partial_files=0
[05:44:28] INFO | 10. Fetch summary | new_rows=0 | successful_pages=0 | requests=1 | retries=0 | partial_saves=0 | elapsed=1.7s
[05:44:28] INFO | 11. Returning slice | rows=58,185 | range=2026-01-01 00:00:00+00:00 -> 2026-07-22 00:40:00+00:00
[05:44:28] INFO | quant.engine | backtest done | bars=58,185 trades=665 final_cash=10748.80 return=7.49% dd=15.25% | 13.9 ms
[05:44:28] INFO | quant.data | cache EXTEND binance PAXGUSDT 15m -> incremental fetch
[05:44:28] INFO | 01. Parameters: market=spot, symbol=PAXGUSDT, interval=15m, start=2026-01-01 00:00:00+00:00, end=2026-07-31 00:00:00+00:00
[05:44:28] INFO | 02. Cache directory: C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\data (cwd=C:\Users\Talal Zah

PAXGUSDT 15m:   0%|          | 0/1 [00:00<?, ?page/s]

[05:44:29] INFO | 08. Completed fetch range 1/1 | rows=0 | total_rows_fetched=0 | requests=1 | retries=0 | elapsed=1.4s
[05:44:30] INFO | 09. Cache consolidated and saved (parquet) | rows=54,435 | range=2025-01-01 00:00:00+00:00 -> 2026-07-22 00:30:00+00:00 | cleared_partial_files=0
[05:44:30] INFO | 10. Fetch summary | new_rows=0 | successful_pages=0 | requests=1 | retries=0 | partial_saves=0 | elapsed=1.6s
[05:44:30] INFO | 11. Returning slice | rows=19,395 | range=2026-01-01 00:00:00+00:00 -> 2026-07-22 00:30:00+00:00
[05:44:30] INFO | quant.engine | backtest done | bars=19,395 trades=606 final_cash=8989.80 return=-10.10% dd=23.19% | 4.9 ms
[05:44:30] INFO | quant.data | cache EXTEND binance PAXGUSDT 1h -> incremental fetch
[05:44:30] INFO | 01. Parameters: market=spot, symbol=PAXGUSDT, interval=1h, start=2026-01-01 00:00:00+00:00, end=2026-07-31 00:00:00+00:00
[05:44:30] INFO | 02. Cache directory: C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\data (cwd=C:\Users\Talal Zahid\

PAXGUSDT 1h:   0%|          | 0/1 [00:00<?, ?page/s]

[05:44:31] INFO | 08. Completed fetch range 1/1 | rows=0 | total_rows_fetched=0 | requests=1 | retries=0 | elapsed=1.6s
[05:44:31] INFO | 09. Cache consolidated and saved (parquet) | rows=13,609 | range=2025-01-01 00:00:00+00:00 -> 2026-07-22 00:00:00+00:00 | cleared_partial_files=0
[05:44:31] INFO | 10. Fetch summary | new_rows=0 | successful_pages=0 | requests=1 | retries=0 | partial_saves=0 | elapsed=1.7s
[05:44:31] INFO | 11. Returning slice | rows=4,849 | range=2026-01-01 00:00:00+00:00 -> 2026-07-22 00:00:00+00:00
[05:44:31] INFO | quant.engine | backtest done | bars=4,849 trades=454 final_cash=8896.99 return=-11.03% dd=23.93% | 1.4 ms


,tf,bars,trades,return%,sharpe,maxDD%,PF
0,1m,290925,730.0,23.9,1.37,14.6,1.073
1,5m,58185,665.0,7.5,0.57,15.2,1.025
2,15m,19395,606.0,-10.1,-0.44,23.2,0.959
3,1h,4849,454.0,-11.0,-0.51,23.9,0.942


## Q6 . Best EMA period + higher-timeframe bias filter
Only take trades in the direction of the HTF EMA (e.g. only long when price is above the 1h EMA).


In [10]:
r = study('q6_ema_htf', 'EMA period x HTF bias',
          strat_space={'ema_period':[20,50,100,200], 'entry_mode':['full_candle'], 'confirm_n':[2],
                       'exit_mode':['none'], 'htf':[None,'1h'], 'htf_ema':[50]})
r[['ema_period','htf']+SHOW].head(12)

[05:45:01] INFO | experiments | experiment 'q6_ema_htf': 8 strategy x 1 cfg = 8 trials | bars=58185
[05:45:02] INFO | experiments | experiment 'q6_ema_htf' -> experiments\results\q6_ema_htf (best sharpe=1.2746)


,ema_period,htf,num_trades,total_return_pct,win_rate_pct,profit_factor,sharpe,max_drawdown_pct
0,20,None,651.0,21.408200,35.944700,1.068860,1.274639,16.309358
1,50,None,665.0,7.488011,34.887218,1.025265,0.569701,15.247606
2,20,1h,600.0,4.536543,34.666667,1.017025,0.420569,23.072482
3,100,1h,601.0,3.320425,34.608985,1.012907,0.347178,21.665305
4,200,1h,629.0,0.544265,34.340223,1.001984,0.187600,24.463202
5,100,None,680.0,-0.343759,34.264706,0.998798,0.139376,18.605691
6,50,1h,603.0,-2.753900,33.996683,0.989154,-0.017177,23.434088
7,200,None,692.0,-12.183789,33.236994,0.958366,-0.572284,25.951337


## Q7 . Stop-loss type
Fixed % vs previous-swing structure stop. (ATR stop: `prepare()` adds `atr_14`; build an ATR
stop-level column and use `sl_mode='ref_col'` against it - an easy extension.)


In [11]:
entry = {'ema_period':[50], 'entry_mode':['full_candle'], 'confirm_n':[2], 'exit_mode':['none']}
r_pct = study('q7_sl_pct', 'fixed % stop', strat_space=entry,
              cfg_space={'sl_mode':['entry_pct'], 'sl_value':[0.3,0.5,0.75,1.0,1.5], 'tp_value':[2.0]})
display(r_pct[['sl_value']+SHOW])
cfg_ref = dataclasses.replace(base_cfg, sl_mode='ref_col', sl_ref_long_col='swing_last_low',
                              sl_ref_short_col='swing_last_high', sl_buffer_pct=0.05, sl_max_ref_risk_pct=2.0)
r_ref = study('q7_sl_swing', 'previous-swing structure stop', strat_space=entry,
              cfg_space={'tp_value':[1.5,2.0,3.0]}, cfg=cfg_ref)
display(r_ref[['tp_value']+SHOW])

[05:45:10] INFO | experiments | experiment 'q7_sl_pct': 1 strategy x 5 cfg = 5 trials | bars=58185


q7_sl_pct:   0%|          | 0/1 [00:00<?, ?grp/s]

[05:45:10] INFO | experiments | experiment 'q7_sl_pct' -> experiments\results\q7_sl_pct (best sharpe=1.3200)


,sl_value,num_trades,total_return_pct,win_rate_pct,profit_factor,sharpe,max_drawdown_pct
0,1.00,257.0,22.398796,36.575875,1.109903,1.320049,21.985749
1,1.50,137.0,12.153651,36.496350,1.124919,1.080303,16.144572
2,0.75,437.0,12.446390,35.469108,1.050218,0.825454,18.015998
3,0.50,967.0,-5.807337,33.919338,0.982190,-0.178114,20.231219
4,0.30,2284.0,-30.881800,32.968476,0.923693,-1.891777,34.831876


[05:45:10] INFO | experiments | experiment 'q7_sl_swing': 1 strategy x 3 cfg = 3 trials | bars=58185


q7_sl_swing:   0%|          | 0/1 [00:00<?, ?grp/s]

[05:45:10] INFO | experiments | experiment 'q7_sl_swing' -> experiments\results\q7_sl_swing (best sharpe=2.0388)


,tp_value,num_trades,total_return_pct,win_rate_pct,profit_factor,sharpe,max_drawdown_pct
0,2.0,238.0,31.122678,37.815126,1.216041,2.038804,11.209063
1,3.0,180.0,-3.849023,24.444444,0.967781,-0.150152,18.386235
2,1.5,353.0,-8.727805,39.660057,0.952251,-0.490032,23.933323


## Q8 . Take-profit type (RR multiple vs fixed %)


In [12]:
r = study('q8_tp_rr', 'RR take-profit', strat_space=entry,
          cfg_space={'sl_value':[0.6], 'tp_mode':['rr'], 'tp_value':[1.0,1.5,2.0,3.0,4.0]})
display(r[['tp_value']+SHOW])
r2 = study('q8_tp_pct', 'fixed-% take-profit', strat_space=entry,
           cfg_space={'sl_value':[0.6], 'tp_mode':['entry_pct'], 'tp_value':[0.5,1.0,2.0,3.0]})
display(r2[['tp_value']+SHOW])

[05:45:20] INFO | experiments | experiment 'q8_tp_rr': 1 strategy x 5 cfg = 5 trials | bars=58185


q8_tp_rr:   0%|          | 0/1 [00:00<?, ?grp/s]

[05:45:20] INFO | experiments | experiment 'q8_tp_rr' -> experiments\results\q8_tp_rr (best sharpe=0.5697)


,tp_value,num_trades,total_return_pct,win_rate_pct,profit_factor,sharpe,max_drawdown_pct
0,2.0,665.0,7.488011,34.887218,1.025265,0.569701,15.247606
1,4.0,380.0,7.403429,21.315789,1.035342,0.566507,21.760758
2,3.0,466.0,3.245195,25.965665,1.014731,0.341198,16.143936
3,1.5,929.0,-6.081372,40.581270,0.982273,-0.189731,17.859630
4,1.0,1263.0,-12.032717,50.277118,0.968227,-0.553647,24.209959


[05:45:20] INFO | experiments | experiment 'q8_tp_pct': 1 strategy x 4 cfg = 4 trials | bars=58185


q8_tp_pct:   0%|          | 0/1 [00:00<?, ?grp/s]

[05:45:20] INFO | experiments | experiment 'q8_tp_pct' -> experiments\results\q8_tp_pct (best sharpe=0.5368)


,tp_value,num_trades,total_return_pct,win_rate_pct,profit_factor,sharpe,max_drawdown_pct
0,3.0,350.0,6.827325,18.000000,1.033766,0.536776,17.550210
1,2.0,473.0,-0.278401,23.890063,0.998788,0.143801,17.080171
2,1.0,823.0,-12.821893,37.424058,0.957763,-0.609882,21.573823
3,0.5,1487.0,-25.455672,54.001345,0.932151,-1.462011,31.207103


## Q9 . Let winners run - trailing stop vs signal-based ride
Trailing stop ratchets on the high-water mark. Signal rides: `exit_mode='ha_flip'` (exit on first
opposite Heikin-Ashi) or `'below_ema'` (exit when a full candle forms beyond a shorter EMA).


In [13]:
cfg_trail = dataclasses.replace(base_cfg, tp_mode='none', trail_mode='pct')
r = study('q9_trailing', 'trailing-stop widths', strat_space=entry,
          cfg_space={'sl_value':[0.6], 'trail_value':[0.3,0.5,0.8,1.2,2.0]}, cfg=cfg_trail)
display(r[['trail_value']+SHOW])
r2 = study('q9_signal_exits', 'ride with HA-flip / shorter-EMA exit',
           strat_space={'ema_period':[100], 'entry_mode':['full_candle'], 'confirm_n':[1,2],
                        'exit_mode':['ha_flip','below_ema'], 'exit_ema':[20,50]},
           cfg_space={'sl_value':[0.6]})
display(r2[['exit_mode','exit_ema','confirm_n']+SHOW])

[05:45:37] INFO | experiments | experiment 'q9_trailing': 1 strategy x 5 cfg = 5 trials | bars=58185


q9_trailing:   0%|          | 0/1 [00:00<?, ?grp/s]

[05:45:37] INFO | experiments | experiment 'q9_trailing' -> experiments\results\q9_trailing (best sharpe=0.1716)


,trail_value,num_trades,total_return_pct,win_rate_pct,profit_factor,sharpe,max_drawdown_pct
0,2.0,289.0,0.246534,22.145329,1.001782,0.171630,18.798956
1,1.2,497.0,-4.291738,28.973843,0.976415,-0.083151,19.294247
2,0.8,868.0,-30.146947,35.599078,0.851255,-1.818577,42.234341
3,0.5,1952.0,-70.780150,34.426230,0.680770,-6.630681,72.475643
4,0.3,4378.0,-85.104280,33.851074,0.689808,-10.411078,85.286628


[05:45:37] INFO | experiments | experiment 'q9_signal_exits': 8 strategy x 1 cfg = 8 trials | bars=58185
[05:45:37] INFO | experiments | experiment 'q9_signal_exits' -> experiments\results\q9_signal_exits (best sharpe=-6.4224)


,exit_mode,exit_ema,confirm_n,num_trades,total_return_pct,win_rate_pct,profit_factor,sharpe,max_drawdown_pct
0,below_ema,50,2,4445.0,-67.593172,19.685039,0.775141,-6.422368,68.209056
1,below_ema,50,1,4726.0,-69.675521,19.932289,0.772042,-6.672060,70.249247
2,below_ema,20,2,9307.0,-90.174531,20.629634,0.685410,-13.478510,90.276235
3,below_ema,20,1,9704.0,-91.040926,20.744023,0.687843,-13.692200,91.125811
4,ha_flip,20,1,23432.0,-99.628650,23.280130,0.603369,-33.028402,99.629578
5,ha_flip,50,1,23432.0,-99.628650,23.280130,0.603369,-33.028402,99.629578
6,ha_flip,50,2,22750.0,-99.579758,23.072527,0.596820,-33.055663,99.580808
7,ha_flip,20,2,22750.0,-99.579758,23.072527,0.596820,-33.055663,99.580808


## Metrics - what to focus on
- **Sharpe / Sortino** - risk-adjusted return; primary ranking. Sortino penalises only downside.
- **Profit factor** (gross win / gross loss) - >1.3 decent, >1.5 good, *after* costs.
- **Max drawdown %** - survivability; big return + 90% DD is untradeable.
- **Win rate + payoff** - low win-rate can still win with high payoff; check `expectancy_per_trade`.
- **num_trades / exposure** - enough trades for significance (the `min_trades` filter).
- **Attribution (Q4)** - an edge concentrated in one session is more believable than a diffuse one.

**Judging a winner:** prefer a *plateau* of similar-scoring neighbours (robust), not a lone spike
(overfit); then **validate out-of-sample** and with realistic spread. Fewer tuned knobs = safer.

> Reality check: raw EMA-cross variants often *lose* on 1m gold because spread eats every trade -
> that's a finding. Watch whether confirmation, full-candle entries, an HTF bias, and a higher
> timeframe push Sharpe positive. If nothing clears costs, 'no edge here' is a valuable cheap answer.
